In [1]:
#@title 1. Setup (auto: latest stable Dify + free Cloudflare tunnel)
import os, json, secrets, subprocess, socket, time, re, shutil

BASE = "/content"
DATA = f"{BASE}/dify-data"
LOGS = f"{DATA}/logs"
os.makedirs(LOGS, exist_ok=True)

PG_PASS = REDIS_PASS = "difyai123456"

_kf = f"{DATA}/keys.json"
KEYS = json.load(open(_kf)) if os.path.exists(_kf) else {
    k: secrets.token_urlsafe(32) for k in ("SECRET_KEY", "DAEMON_KEY", "INNER_KEY")}
json.dump(KEYS, open(_kf, "w"))

_tags = subprocess.check_output("git ls-remote --tags --refs https://github.com/langgenius/dify.git",
                                shell=True, text=True)
DIFY_VERSION = max(re.findall(r"refs/tags/(\d+\.\d+\.\d+)$", _tags, re.M),
                   key=lambda v: tuple(map(int, v.split("."))))
print("Latest Dify:", DIFY_VERSION)

os.environ.update(DIFY_VERSION=DIFY_VERSION,
                  PG_PASS=PG_PASS, REDIS_PASS=REDIS_PASS,
                  PATH="/usr/local/go/bin:" + os.environ["PATH"])

def bg(name, cmd, env=None, cwd=None):
    """Start a long-running process in the background, logging to LOGS/name.log."""
    log = open(f"{LOGS}/{name}.log", "a")
    return subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                            stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

def wait_port(port, name, timeout=300):
    t = time.time()
    while time.time() - t < timeout:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                print(f"{name} is up on :{port}"); return True
        time.sleep(2)
    print(f"{name} did not start on :{port} — check {LOGS}/{name}.log"); return False

print("Settings saved.")

Latest Dify: 1.17.1
Settings saved.


In [2]:
%%bash
#@title 2. Install system packages (safe to re-run)
set -e
export DEBIAN_FRONTEND=noninteractive

fetch() { curl -fsSL -o "$2.tmp" "$1" && mv -f "$2.tmp" "$2"; }

if ! command -v pg_ctlcluster >/dev/null || [ ! -d /usr/lib/postgresql/16 ] || ! command -v nginx >/dev/null || ! command -v redis-server >/dev/null; then
  apt-get -qq update 2>/dev/null || true
  apt-get -qq install -y postgresql-common ca-certificates curl > /dev/null
  /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh -y > /dev/null 2>&1
  apt-get -qq install -y postgresql-16 postgresql-16-pgvector redis-server nginx libmagic1 > /dev/null
fi

if ! node -v 2>/dev/null | grep -q '^v24'; then
  curl -fsSL https://deb.nodesource.com/setup_24.x | bash - > /dev/null 2>&1
  apt-get -qq install -y nodejs > /dev/null
fi

command -v uv >/dev/null || pip -q install -U uv

if ! command -v cloudflared >/dev/null; then
  fetch https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 /usr/local/bin/cloudflared
  chmod +x /usr/local/bin/cloudflared
fi

if ! command -v crane >/dev/null; then
  fetch https://github.com/google/go-containerregistry/releases/latest/download/go-containerregistry_Linux_x86_64.tar.gz /tmp/crane.tgz
  tar -xzf /tmp/crane.tgz -C /usr/local/bin crane
fi

if [ ! -x /usr/local/go/bin/go ]; then
  GO=$(curl -fsSL 'https://go.dev/VERSION?m=text' | head -1 || true)
  fetch "https://go.dev/dl/${GO}.linux-amd64.tar.gz" /tmp/go.tgz
  tar -xzf /tmp/go.tgz -C /usr/local
fi

echo "node $(node -v) | $(uv --version) | $(/usr/local/go/bin/go version) | $(cloudflared --version 2>&1 | head -1)"

node v24.21.0 | uv 0.12.21 (x86_64-unknown-linux-gnu) | go version go1.27.1 linux/amd64 | cloudflared version 2026.10.0 (built 2026-10-05-17:37 UTC)


In [3]:
%%bash
#@title 3. Dify API (source + Python deps, ~5 min)
set -e
cd /content
[ -d dify ] || git clone -q --depth 1 -b "$DIFY_VERSION" https://github.com/langgenius/dify.git
cd dify/api
export UV_PYTHON=3.12
uv sync --frozen --no-dev -q
uv run --no-sync python -m nltk.downloader -q -d /content/nltk_data punkt_tab averaged_perceptron_tagger_eng stopwords
echo "API ready: $(uv run --no-sync python --version)"

API ready: Python 3.12.3


Note: switching to '8387590ace4a094de812b7847fc6a4c3a27cd52b'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Error loading punkt_tab: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A 

In [4]:
%%bash
#@title 4. Dify Web (prebuilt from official image)
set -e
WEB=/content/dify-web
SERVER=$WEB/app/targets/next/web/server.js
if [ ! -f "$SERVER" ]; then
  mkdir -p $WEB
  if crane export --platform linux/amd64 "langgenius/dify-web:$DIFY_VERSION" - \
       | tar -x --no-same-owner -C $WEB app 2>/dev/null && [ -f "$SERVER" ]; then
    echo "Extracted prebuilt web from Docker Hub."
  else
    echo "Docker Hub pull failed (rate limit?) → building web from source, ~15 min..."
    cd /content/dify
    corepack enable && corepack install
    pnpm install --frozen-lockfile --ignore-scripts
    cd web && NODE_OPTIONS=--max-old-space-size=6144 NEXT_TELEMETRY_DISABLED=1 pnpm build
    mkdir -p $WEB/app/targets/next
    cp -r .next/standalone/. $WEB/app/targets/next/
    cp -r public $WEB/app/targets/next/web/public
    cp -r .next/static $WEB/app/targets/next/web/.next/static
  fi
fi
ls -la "$SERVER"

Extracted prebuilt web from Docker Hub.
-rw-r--r-- 1 root root 7762 Sep 10 09:26 /content/dify-web/app/targets/next/web/server.js


In [5]:
%%bash
#@title 5. Plugin daemon (build with Go, ~3 min)
set -e
export GOTOOLCHAIN=auto
mkdir -p /content/bin
cd /content

PLUGIN_DAEMON_VERSION=$(grep -o 'dify-plugin-daemon:[0-9.]*' dify/docker/docker-compose.yaml | head -1 | cut -d: -f2)
echo "Plugin daemon: $PLUGIN_DAEMON_VERSION"
[ -d dify-plugin-daemon ] || git clone -q --depth 1 -b "$PLUGIN_DAEMON_VERSION" https://github.com/langgenius/dify-plugin-daemon.git
cd dify-plugin-daemon
[ -x /content/bin/plugin-daemon ] || go build -o /content/bin/plugin-daemon ./cmd/server
[ -x /content/bin/plugin-cli ]    || go build -o /content/bin/plugin-cli ./cmd/commandline
uv python install 3.12 -q

ls -la /content/bin

Plugin daemon: 0.6.10
total 172068
drwxr-xr-x 2 root root      4096 Oct  6 14:29 .
drwxr-xr-x 1 root root      4096 Oct  6 14:24 ..
-rwxr-xr-x 1 root root  56419418 Oct  6 14:29 plugin-cli
-rwxr-xr-x 1 root root 119766394 Oct  6 14:29 plugin-daemon


Note: switching to '1310a18b2f6bc6f18768a0a6265484830891433c'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

go: downloading github.com/kelseyhightower/envconfig v1.4.0
go: downloading github.com/getsentry/sentry-go v0.30.0
go: downloading github.com/gin-contrib/gzip v1.2.6
go: downloading github.com/gin-gonic/gin v1.12.0
go: downloading github.com/langgenius/dify-cloud-kit v0.1.3
go: downloading github.com/prometheus/client_golang v1.23.2
go: downloading go.opentelemetry.io/contrib/instrumentation/github.com/gin-gonic/gin/

In [6]:
%%bash
#@title 6. Start PostgreSQL + Redis
set -e
service postgresql start
su postgres -c "psql -q -c \"ALTER USER postgres PASSWORD '$PG_PASS';\""
for db in dify dify_plugin; do
  su postgres -c "psql -tAc \"SELECT 1 FROM pg_database WHERE datname='$db'\" | grep -q 1 || createdb $db"
done
redis-cli -a "$REDIS_PASS" ping 2>/dev/null | grep -q PONG || \
  redis-server --daemonize yes --requirepass "$REDIS_PASS" --save "" --appendonly no
sleep 1 && redis-cli -a "$REDIS_PASS" ping 2>/dev/null

 * Starting PostgreSQL 16 database server
   ...done.
PONG


In [7]:
#@title 7. Start free Cloudflare tunnel (*.trycloudflare.com)
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
time.sleep(1)
open(f"{LOGS}/cloudflared.log", "w").close()

bg("cloudflared", "cloudflared tunnel --no-autoupdate --url http://localhost:8080")
PUBLIC_URL = None
for _ in range(60):
    found = [u for u in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{LOGS}/cloudflared.log").read())
             if not u.startswith("https://api.")]
    if found:
        PUBLIC_URL = found[0]; break
    time.sleep(1)
assert PUBLIC_URL, f"No tunnel URL yet — see {LOGS}/cloudflared.log"

HOST = PUBLIC_URL.removeprefix("https://")
print("Public URL:", PUBLIC_URL)

Public URL: https://meyer-unable-paragraphs-howard.trycloudflare.com


In [8]:
#@title Start Dify + free Cloudflare tunnel
import psutil

def _free(p):
    with socket.socket() as s: return s.connect_ex(("127.0.0.1", p)) != 0

def _kill_port(port):
    """Kill whatever listens on a port, whatever its process name (Next.js renames itself)."""
    for c in psutil.net_connections(kind="tcp"):
        if c.laddr and c.laddr.port == port and c.status == psutil.CONN_LISTEN and c.pid:
            try:
                p = psutil.Process(c.pid)
                for ch in p.children(recursive=True): ch.kill()
                p.kill()
            except psutil.NoSuchProcess:
                pass

for pat in ["cloudflared tunnel", "gunicorn", "celery", "plugin-daemon", "server.js", "next-server"]:
    subprocess.run(["pkill", "-9", "-f", pat])
subprocess.run(["nginx", "-s", "stop"], stderr=subprocess.DEVNULL)
for p in (5001, 5002, 5003, 3000):
    for _ in range(15):
        if _free(p): break
        _kill_port(p)
        time.sleep(1)
    else:
        raise RuntimeError(f"Port {p} is still busy: {[c.pid for c in psutil.net_connections('tcp') if c.laddr and c.laddr.port == p]}")

NGINX_PORT = next(p for p in range(8090, 8200) if _free(p))

open(f"{LOGS}/cloudflared.log", "w").close()
bg("cloudflared", f"cloudflared tunnel --no-autoupdate --url http://127.0.0.1:{NGINX_PORT}")
PUBLIC_URL = None
for _ in range(60):
    found = [u for u in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{LOGS}/cloudflared.log").read())
             if not u.startswith("https://api.")]
    if found:
        PUBLIC_URL = found[0]; break
    time.sleep(1)
assert PUBLIC_URL, f"No tunnel URL yet — see {LOGS}/cloudflared.log"
HOST = PUBLIC_URL.removeprefix("https://")
print("Public URL:", PUBLIC_URL, "| nginx port:", NGINX_PORT)

API_DIR = f"{BASE}/dify/api"
WEB_DIR = f"{BASE}/dify-web/app/targets/next/web"
PD_DIR  = f"{DATA}/plugin_daemon"
os.makedirs(PD_DIR, exist_ok=True)

API_ENV = dict(
    SECRET_KEY=KEYS["SECRET_KEY"], DEPLOY_ENV="PRODUCTION", EDITION="SELF_HOSTED",
    LOG_LEVEL="INFO", FLASK_APP="app.py", NLTK_DATA=f"{BASE}/nltk_data", PYTHONIOENCODING="utf-8",
    CONSOLE_API_URL=PUBLIC_URL, CONSOLE_WEB_URL=PUBLIC_URL, SERVICE_API_URL=PUBLIC_URL,
    APP_API_URL=PUBLIC_URL, APP_WEB_URL=PUBLIC_URL, FILES_URL=PUBLIC_URL, TRIGGER_URL=PUBLIC_URL,
    INTERNAL_FILES_URL="http://127.0.0.1:5001",
    DB_TYPE="postgresql", DB_USERNAME="postgres", DB_PASSWORD=PG_PASS,
    DB_HOST="127.0.0.1", DB_PORT="5432", DB_DATABASE="dify",
    REDIS_HOST="127.0.0.1", REDIS_PORT="6379", REDIS_PASSWORD=REDIS_PASS, REDIS_DB="0",
    CELERY_BROKER_URL=f"redis://:{REDIS_PASS}@127.0.0.1:6379/1",
    STORAGE_TYPE="opendal", OPENDAL_SCHEME="fs", OPENDAL_FS_ROOT=f"{DATA}/storage",
    VECTOR_STORE="pgvector", PGVECTOR_HOST="127.0.0.1", PGVECTOR_PORT="5432",
    PGVECTOR_USER="postgres", PGVECTOR_PASSWORD=PG_PASS, PGVECTOR_DATABASE="dify",
    PLUGIN_DAEMON_URL="http://127.0.0.1:5002", PLUGIN_DAEMON_KEY=KEYS["DAEMON_KEY"],
    INNER_API_KEY_FOR_PLUGIN=KEYS["INNER_KEY"],
    MARKETPLACE_ENABLED="true", MARKETPLACE_API_URL="https://marketplace.dify.ai",
    ENDPOINT_URL_TEMPLATE=f"{PUBLIC_URL}/e/{{hook_id}}",
    ENABLE_COLLABORATION_MODE="false",
)

PD_ENV = dict(
    SERVER_HOST="127.0.0.1", SERVER_PORT="5002", SERVER_KEY=KEYS["DAEMON_KEY"], GIN_MODE="release",
    PLATFORM="local", DIFY_INNER_API_URL="http://127.0.0.1:5001", DIFY_INNER_API_KEY=KEYS["INNER_KEY"],
    DB_TYPE="postgresql", DB_USERNAME="postgres", DB_PASSWORD=PG_PASS, DB_HOST="127.0.0.1",
    DB_PORT="5432", DB_DATABASE="dify_plugin", DB_SSL_MODE="disable",
    REDIS_HOST="127.0.0.1", REDIS_PORT="6379", REDIS_PASSWORD=REDIS_PASS, REDIS_DB="0", REDIS_USE_SSL="false",
    PLUGIN_STORAGE_TYPE="local", PLUGIN_STORAGE_LOCAL_ROOT=f"{PD_DIR}/storage",
    PLUGIN_INSTALLED_PATH="plugin", PLUGIN_WORKING_PATH=f"{PD_DIR}/cwd",
    PLUGIN_PACKAGE_CACHE_PATH="plugin_packages", PLUGIN_MEDIA_CACHE_PATH="assets",
    PERSISTENCE_STORAGE_PATH="persistence",
    PLUGIN_REMOTE_INSTALLING_HOST="127.0.0.1", PLUGIN_REMOTE_INSTALLING_PORT="5003",
    PYTHON_INTERPRETER_PATH=subprocess.check_output("uv python find 3.12", shell=True, text=True).strip(),
    UV_PATH=shutil.which("uv"), PYTHON_ENV_INIT_TIMEOUT="300",
    FORCE_VERIFYING_SIGNATURE="true", PLUGIN_MAX_EXECUTION_TIMEOUT="600",
    S3_USE_AWS="false", PIP_MIRROR_AUTO_DETECT="false",
)

WEB_ENV = dict(
    NODE_ENV="production", PORT="3000", HOSTNAME="127.0.0.1", NEXT_TELEMETRY_DISABLED="1",
    NEXT_PUBLIC_DEPLOY_ENV="PRODUCTION", NEXT_PUBLIC_EDITION="SELF_HOSTED",
    NEXT_PUBLIC_API_PREFIX=f"{PUBLIC_URL}/console/api",
    NEXT_PUBLIC_PUBLIC_API_PREFIX=f"{PUBLIC_URL}/api",
    NEXT_PUBLIC_WEB_PREFIX=PUBLIC_URL,
    NEXT_PUBLIC_MARKETPLACE_API_PREFIX="https://marketplace.dify.ai/api/v1",
    NEXT_PUBLIC_MARKETPLACE_URL_PREFIX="https://marketplace.dify.ai",
    NEXT_PUBLIC_SOCKET_URL=f"wss://{HOST}",
    NEXT_PUBLIC_ENABLE_WEBSITE_JINAREADER="true", NEXT_PUBLIC_ENABLE_WEBSITE_FIRECRAWL="true",
    NEXT_PUBLIC_ENABLE_WEBSITE_WATERCRAWL="true",
)

QUEUES = ("api_token,dataset,dataset_summary,priority_dataset,priority_pipeline,pipeline,mail,ops_trace,"
          "app_deletion,app_rbac,plugin,workflow_storage,conversation,workflow,schedule_poller,schedule_executor,"
          "triggered_workflow_dispatcher,trigger_refresh_publisher,trigger_refresh_executor,retention,"
          "workflow_based_app_execution")

print("▶ Database migrations (first run takes a minute)...")
r = subprocess.run("uv run --no-sync flask upgrade-db", shell=True, cwd=API_DIR,
                   env={**os.environ, **API_ENV}, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:]) if r.returncode else print("  ✅ migrations done")

print("▶ Plugin daemon migrations...")
subprocess.run(f"{BASE}/bin/plugin-cli migrate", shell=True, cwd=PD_DIR, env={**os.environ, **PD_ENV}, check=True)

print("▶ Starting services...")
bg("api", "uv run --no-sync gunicorn --no-control-socket --bind 127.0.0.1:5001 --workers 1 "
          "--worker-class geventwebsocket.gunicorn.workers.GeventWebSocketWorker "
          "--worker-connections 10 --timeout 360 app:socketio_app", API_ENV, API_DIR)
bg("worker", f"uv run --no-sync celery -A celery_entrypoint.celery worker -P gevent -c 1 "
             f"--max-tasks-per-child 50 --loglevel INFO -Q {QUEUES}", API_ENV, API_DIR)
bg("beat", "uv run --no-sync celery -A app.celery beat --loglevel INFO", API_ENV, API_DIR)
bg("plugin_daemon", f"{BASE}/bin/plugin-daemon", PD_ENV, PD_DIR)
bg("web", "node server.js", WEB_ENV, WEB_DIR)

PROXY = """proxy_set_header Host $host;
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header X-Forwarded-Proto https;
      proxy_http_version 1.1;
      proxy_buffering off;
      proxy_read_timeout 3600s;
      proxy_send_timeout 3600s;"""
api_loc = lambda p: f"    location {p} {{\n      proxy_pass http://127.0.0.1:5001;\n      proxy_set_header Connection \"\";\n      {PROXY}\n    }}\n"
NGINX = (f"server {{\n    listen 127.0.0.1:{NGINX_PORT};\n    client_max_body_size 100M;\n"
         + "".join(api_loc(p) for p in ["/console/api", "/api", "/v1", "/openapi", "/files", "/mcp", "/triggers"])
         + f"""    location /socket.io/ {{
      proxy_pass http://127.0.0.1:5001;
      {PROXY}
      proxy_set_header Upgrade $http_upgrade;
      proxy_set_header Connection "upgrade";
    }}
    location /e/ {{
      proxy_pass http://127.0.0.1:5002;
      proxy_set_header Dify-Hook-Url https://$host$request_uri;
      proxy_set_header Connection "";
      {PROXY}
    }}
    location / {{
      proxy_pass http://127.0.0.1:3000;
      proxy_set_header Connection "";
      {PROXY}
    }}
}}
""")
open("/etc/nginx/sites-available/dify", "w").write(NGINX)
subprocess.run("rm -f /etc/nginx/sites-enabled/default && ln -sf /etc/nginx/sites-available/dify /etc/nginx/sites-enabled/dify", shell=True)
r = subprocess.run("nginx -t && nginx", shell=True, capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("nginx failed:\n" + r.stderr)

ok = all([wait_port(5001, "api"), wait_port(5002, "plugin_daemon"), wait_port(3000, "web"), wait_port(NGINX_PORT, "nginx")])

import urllib.request

_af = f"{DATA}/admin.json"
_had_admin_file = os.path.exists(_af)
ADMIN = json.load(open(_af)) if os.path.exists(_af) else {
    "email": "admin@example.com", "name": "admin", "password": "Dify" + secrets.token_hex(6)}

json.dump(ADMIN, open(_af, "w"))

for _ in range(60):
    try:
        status = json.load(urllib.request.urlopen("http://127.0.0.1:5001/console/api/setup"))
        break
    except Exception:
        time.sleep(2)

if status.get("step") != "finished":
    req = urllib.request.Request("http://127.0.0.1:5001/console/api/setup", method="POST",
                                 data=json.dumps({**ADMIN, "language": "en-US"}).encode(),
                                 headers={"Content-Type": "application/json"})
    urllib.request.urlopen(req)
    print("admin account created")

elif not _had_admin_file:
    ADMIN = {"email": "(the email you registered)", "password": "(your password)"}
    os.remove(_af)

from IPython.display import display, HTML
if ok:
    display(HTML(f"""<h2>🎉 Dify is running</h2>
    <p>Open: <a href="{PUBLIC_URL}/signin" target="_blank">{PUBLIC_URL}</a></p>
    <p>Email: <code>{ADMIN['email']}</code> &nbsp; Password: <code>{ADMIN['password']}</code></p>
    <p>API base URL for your apps: <code>{PUBLIC_URL}/v1</code></p>"""))

Public URL: https://suppose-acknowledge-combinations-get.trycloudflare.com | nginx port: 8090
▶ Database migrations (first run takes a minute)...
  ✅ migrations done
▶ Plugin daemon migrations...
▶ Starting services...
api is up on :5001
plugin_daemon is up on :5002
web is up on :3000
nginx is up on :8090
admin account created


In [10]:
#@title 🦙 Local model server (Ollama or vLLM) — optional, needs a GPU runtime
#@markdown Run this **after** Dify is started. Use *Runtime → Change runtime type → T4 GPU* (or better) first.
ENGINE = "ollama" #@param ["ollama", "vllm"]
#@markdown **Ollama**: any tag from ollama.com/library. The embedding model is for Knowledge (RAG); leave empty to skip.
OLLAMA_MODEL = "qwen2.5:7b"          #@param {type:"string"}
OLLAMA_EMBED_MODEL = "nomic-embed-text" #@param {type:"string"}
#@markdown **vLLM**: a Hugging Face model id (small models fit a free T4; vLLM is happiest on L4/A100).
VLLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct" #@param {type:"string"}
VLLM_MAX_LEN = 4096 #@param {type:"integer"}
HF_TOKEN = "" #@param {type:"string"}

import urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none — it will run on CPU (very slow; Ollama only)")

def _sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout

if ENGINE == "ollama":
    if not shutil.which("ollama"):
        print("▶ Installing Ollama...")
        _sh("apt-get -qq install -y zstd pciutils lshw > /dev/null && curl -fsSL https://ollama.com/install.sh | sh")
    subprocess.run(["pkill", "-9", "-f", "ollama serve"])
    time.sleep(1)
    bg("ollama", "ollama serve", {"OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "-1",
                                  "OLLAMA_MODELS": f"{BASE}/ollama-models"})
    wait_port(11434, "ollama", 60)
    for m in filter(None, [OLLAMA_MODEL.strip(), OLLAMA_EMBED_MODEL.strip()]):
        print(f"▶ Pulling {m} (first time can take a few minutes)...")
        _sh(f"ollama pull {m}")
        print(f"  ✅ {m}")
    MODEL_SETUP = f"""
<b>In Dify:</b> avatar (top-right) → <b>Settings → Model Provider</b> → install <b>Ollama</b> → <b>Add Model</b>:
<ul>
<li>Model Type: <code>LLM</code> · Model Name: <code>{OLLAMA_MODEL}</code> · Base URL: <code>http://127.0.0.1:11434</code></li>
""" + (f"""<li>(for Knowledge) Model Type: <code>Text Embedding</code> · Model Name: <code>{OLLAMA_EMBED_MODEL}</code> · Base URL: <code>http://127.0.0.1:11434</code></li>""" if OLLAMA_EMBED_MODEL.strip() else "") + "</ul>"

else:
    VENV = f"{BASE}/vllm-env"
    if not os.path.exists(f"{VENV}/bin/vllm"):
        print("▶ Installing vLLM (~5–10 min, one time)...")
        _sh(f"uv venv -q --python 3.12 {VENV} && uv pip install -q --python {VENV}/bin/python vllm")
    subprocess.run(["pkill", "-9", "-f", "vllm serve"])
    time.sleep(2)
    SERVED = VLLM_MODEL.split("/")[-1]
    major = int(subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True,
                               capture_output=True, text=True).stdout.strip().split(".")[0] or 0)
    dtype = "half" if major < 8 else "auto"   # T4 (7.5) has no bfloat16
    env = {"HF_HOME": f"{BASE}/hf-cache"}
    if HF_TOKEN.strip(): env["HF_TOKEN"] = HF_TOKEN.strip()
    bg("vllm", f"{VENV}/bin/vllm serve {VLLM_MODEL} --host 127.0.0.1 --port 8000 --served-model-name {SERVED} "
               f"--dtype {dtype} --max-model-len {VLLM_MAX_LEN} --gpu-memory-utilization 0.85", env)
    print("▶ Downloading + loading model (several minutes)...")
    for _ in range(450):   # up to 15 min
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5); break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError(f"vLLM didn't come up — check {LOGS}/vllm.log")
    print("vLLM is serving", SERVED)
    MODEL_SETUP = f"""
<b>In Dify:</b> avatar (top-right) → <b>Settings → Model Provider</b> → install <b>OpenAI-API-compatible</b> → <b>Add Model</b>:
<ul><li>Model Type: <code>LLM</code> · Model Name: <code>{SERVED}</code> ·
API endpoint URL: <code>http://127.0.0.1:8000/v1</code> · API Key: <code>EMPTY</code> ·
Context size: <code>{VLLM_MAX_LEN}</code></li></ul>"""

from IPython.display import display, HTML
display(HTML(f"<h3>🦙 {ENGINE} is running in the background</h3>{MODEL_SETUP}"
             f"<p>Then pick it in any app's model selector. Dify: <a href='{PUBLIC_URL}' target='_blank'>{PUBLIC_URL}</a></p>"))

GPU: Tesla T4, 15360 MiB
▶ Installing Ollama...
ollama is up on :11434
▶ Pulling qwen2.5:7b (first time can take a few minutes)...
  ✅ qwen2.5:7b
▶ Pulling nomic-embed-text (first time can take a few minutes)...
  ✅ nomic-embed-text


In [9]:
#@title Status & logs
SERVICE = "cloudflared" #@param ["api", "worker", "beat", "plugin_daemon", "web", "cloudflared", "ollama", "vllm"]
LINES = 40 #@param {type:"integer"}
import urllib.request
def _up(port):
    with socket.socket() as s: return s.connect_ex(("127.0.0.1", port)) == 0
for port, name in [(5432, "postgres"), (6379, "redis"), (5001, "api"), (5002, "plugin_daemon"),
                   (3000, "web"), (NGINX_PORT, "nginx"),
                   (11434, "ollama"), (8000, "vllm")]:
    print(f"{'🟢' if _up(port) else '🔴'} {name:14} :{port}")
try:
    urllib.request.urlopen(PUBLIC_URL + "/console/api/setup", timeout=15)
    print(f"🟢 {'tunnel':14} {PUBLIC_URL}")
except Exception as e:
    print(f"🔴 {'tunnel':14} {PUBLIC_URL}  ({e})")
print(f"\nLogin: {ADMIN['email']} / {ADMIN['password']}")
print(f"\n--- last {LINES} lines of {SERVICE}.log ---")
print(subprocess.run(["tail", "-n", str(LINES), f"{LOGS}/{SERVICE}.log"], capture_output=True, text=True).stdout)

🟢 postgres       :5432
🟢 redis          :6379
🟢 api            :5001
🟢 plugin_daemon  :5002
🟢 web            :3000
🟢 nginx          :8090
🔴 ollama         :11434
🔴 vllm           :8000
🟢 tunnel         https://suppose-acknowledge-combinations-get.trycloudflare.com

Login: admin@example.com / Difyb4f8f414a381

--- last 40 lines of cloudflared.log ---
2026-10-06T14:30:16Z INF +--------------------------------------------------------------------------------------------+
2026-10-06T14:30:16Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-06T14:30:16Z INF |  https://suppose-acknowledge-combinations-get.trycloudflare.com                            |
2026-10-06T14:30:16Z INF +--------------------------------------------------------------------------------------------+
2026-10-06T14:30:16Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/clou

In [ ]:
import time

while True:
    time.sleep(30)


In [ ]:
#@title 10. Backup to Google Drive
#@markdown Dumps both databases + uploaded files + installed plugins to `MyDrive/dify-backup/`.

from google.colab import drive
drive.mount("/content/drive")
dst = "/content/drive/MyDrive/dify-backup"
os.makedirs(dst, exist_ok=True)
for db in ("dify", "dify_plugin"):
    subprocess.run(f"su postgres -c 'pg_dump -Fc {db}' > {dst}/{db}.dump", shell=True, check=True)
subprocess.run(f"tar -czf {dst}/files.tgz -C {DATA} storage plugin_daemon/storage keys.json", shell=True, check=True)
print("Backup written to", dst)